# 🛡️ ARMOR — Complete Experiment Suite (Google Colab)
### Adaptive Relearning-Resistant Multimodal Unlearning

**Target: ≤ 60 minutes on T4 GPU** | Uses `ARMOR_FAST=1` for maximum speed

| Phase | Methods | Est. Time |
|-------|---------|-----------|
| Setup | GPU · Deps · Model cache | ~5 min |
| Core Baselines | GA · NPO · NPO+SAM · RMU · Task Vector | ~30 min |
| Relearning Attack | Compare relearning resistance | ~4 min |
| ARMOR Modules | 9 methods (3 batched cells) | ~12 min |
| Extensions | MultiTask · DP · LLaVA · MUSE | *(optional)* |
| Results | Table · Chart · Privacy audit | ~2 min |

> **Before running:** Runtime → Change runtime type → **GPU (T4)**


In [1]:
# ── 0. GPU Check & Timing  ⏱ ~10s ────────────────────────────────────────────
import subprocess, sys, os, time
NOTEBOOK_START = time.time()

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' in gpu_info.stdout:
    print('✅ GPU detected:')
    for line in gpu_info.stdout.split('\n')[:12]:
        print(line)
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → GPU')

import torch
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU     : {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)')


✅ GPU detected:
Sat Jun 27 07:56:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [2]:
# ── 1. Mount Google Drive  ⏱ ~30s ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ARMOR_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Outputs → {DRIVE_DIR}')


Mounted at /content/drive
✅ Outputs → /content/drive/MyDrive/ARMOR_outputs


In [3]:
# ── 2. Clone / Update ARMOR  ⏱ ~1 min ────────────────────────────────────────
REPO_URL = 'https://github.com/Angrajkarn/-ARMOR-Adaptive-Relearning-resistant-Multimodal-Unlearning.git'
REPO_DIR = '/content/ARMOR'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
!git log --oneline -3
print(f'\n✅ Working dir: {os.getcwd()}')


Cloning into '/content/ARMOR'...
remote: Enumerating objects: 553, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (352/352), done.
remote: Total 553 (delta 289), reused 418 (delta 157), pack-reused 0 (from 0)
Receiving objects: 100% (553/553), 1.57 MiB | 12.86 MiB/s, done.
Resolving deltas: 100% (289/289), done.
8a08ab6 (HEAD -> main, origin/main, origin/HEAD) Optimize HDI get_activations by removing inner-hook sync points and fix win32 subprocess encoding deadlocks
2022d9e Fix only Tensors of floating point dtype can require gradients error in lora unlearn remove_and_restore and causal surgery
0162fbc Fix crashes and memory leaks in Continual, Multitask, RLACE-RMU, HDI, and Modular LoRA unlearners

✅ Working dir: /content/ARMOR


In [4]:
# ── 3. Install Dependencies  ⏱ ~3 min ────────────────────────────────────────
import subprocess

pkgs = [
    'transformers>=4.40.0', 'peft>=0.10.0', 'datasets>=2.18.0',
    'accelerate>=0.28.0', 'bitsandbytes>=0.43.0', 'trl>=0.8.0',
    'rouge-score', 'scipy', 'scikit-learn', 'pandas', 'matplotlib',
    'Pillow>=10.0.0', 'torchvision>=0.16.0', 'opacus>=1.4.0',
]

result = subprocess.run(['pip', 'install', '-q'] + pkgs, capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('✅ All packages installed')

# Flash Attention (A100/H100 only — skip errors on T4)
try:
    subprocess.run(['pip', 'install', '-q', 'flash-attn', '--no-build-isolation'],
                   capture_output=True, timeout=90)
    print('✅ Flash Attention installed')
except Exception:
    print('ℹ️  Flash Attention N/A on this GPU (OK)')


✅ All packages installed
ℹ️  Flash Attention N/A on this GPU (OK)


In [5]:
# ── 4. HuggingFace Login  ⏱ ~10s ─────────────────────────────────────────────
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = ''
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged in via Colab Secrets')
except Exception:
    print('⚠️  No HF_TOKEN in Colab Secrets.')
    HF_TOKEN = input('HuggingFace token (Enter to skip for Mistral): ').strip()
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print('✅ Logged in')
    else:
        print('ℹ️  Proceeding without token (Mistral works without one)')


✅ Logged in via Colab Secrets


In [6]:
# ── 5. Configuration  ⏱ ~5s ──────────────────────────────────────────────────
import sys, os, time
sys.path.insert(0, '/content/ARMOR')

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL = 'mistral-7b'    # or 'llama2-7b' (needs HF token) or 'debug' (CPU test)

# ── Speed: ARMOR_FAST env var (inherited by all subprocesses) ─────────────────
# Automatically sets: retain=200 samples, fp16 autocast, epochs=1
os.environ['ARMOR_FAST'] = '1'

# ── Baselines Control ────────────────────────────────────────────────────────
# Toggle execution of individual baselines to save time.
# If False, pre-cached results from the codebase will be used in summaries/plots.
RUN_GA          = True
RUN_NPO         = True
RUN_NPO_SAM     = True
RUN_RMU         = True
RUN_TASK_VECTOR = True

# ── Common CLI args ───────────────────────────────────────────────────────────
FAST = '--no-rouge'            # skip ROUGE (saves ~1 min/method)
HF   = f'--hf-token {HF_TOKEN}' if HF_TOKEN else ''
Q    = '--qlora'
D    = f'--output-dir {DRIVE_DIR}'

print(f'Model      : {MODEL}')
print(f'ARMOR_FAST : 1 (retain=200, fp16, epochs=1)')
print(f'Output     : {DRIVE_DIR}')
print(f'\n⏱  Target: ≤ 60 min on T4')


Model      : mistral-7b
ARMOR_FAST : 1 (retain=200, fp16, epochs=1)
Output     : /content/drive/MyDrive/ARMOR_outputs

⏱  Target: ≤ 60 min on T4


---
## 📊 Section 1 — Core Baselines (~30 min)
These 5 methods form the paper's main comparison table.


In [7]:
# ── 6. Gradient Ascent (GA)  ⏱ ~5 min ────────────────────────────────────────
t0 = time.time()
if RUN_GA:
    !python scripts/run_baseline_ga.py \
        --model {MODEL} {Q} {HF} --fast {FAST} \
        --run-mia \
        --output-dir {DRIVE_DIR}/ga
    print(f'\n✅ GA done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ GA execution skipped (RUN_GA=False). Using pre-computed results.')


[fast] Speed preset active: retain=200, fp16=True, rouge_tokens=32
  ARMOR — Gradient Ascent Baseline
  Model  : mistralai/Mistral-7B-v0.1
  Device : cuda
  Debug  : False
  fp16   : True
  Retain : 200
[data] Loading TOFU — forget=forget01, retain=retain99
README.md: 100% 4.14k/4.14k [00:00<00:00, 283kB/s]
forget01.json: 100% 11.7k/11.7k [00:00<00:00, 17.4MB/s]
Generating train split: 100% 40/40 [00:00<00:00, 252.06 examples/s]
retain99.json: 100% 1.07M/1.07M [00:00<00:00, 54.3MB/s]
Generating train split: 100% 3960/3960 [00:00<00:00, 206659.66 examples/s]
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
config.json: 100% 571/571 [00:00<00:00, 2.22MB/s]
tokenizer_config.json: 100% 996/996 [00:00<00:00, 4.41MB/s]
tokenizer.json: 100% 1.80M/1.80M [00:00<00:00, 49.4MB/s]
tokenizer.model: 100% 493k/493k [00:01<00:00, 282kB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 1.94MB/s]
model.saf

In [8]:
# ── 7. NPO Baseline  ⏱ ~5 min ────────────────────────────────────────────────
t0 = time.time()
if RUN_NPO:
    !python scripts/run_baseline_npo.py \
        --model {MODEL} {Q} {HF} --fast {FAST} \
        --run-mia \
        --output-dir {DRIVE_DIR}/npo
    print(f'\n✅ NPO done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ NPO execution skipped (RUN_NPO=False). Using pre-computed results.')


[fast] Speed preset active: retain=200, fp16=True, rouge_tokens=32
  ARMOR — NPO Baseline
  Model  : mistralai/Mistral-7B-v0.1
  Device : cuda
  β      : 0.1
  fp16   : True
  Retain : 200
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:52,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:58<00:00,  4.97it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)
[model] NPO reference: using base weights (LoRA-disabled mode)

[main] Pre-unlearning evaluation:

[e

In [9]:
# ── 8. NPO+SAM — ARMOR Core  ⏱ ~10 min ──────────────────────────────────────
# sam-every=4 cuts SAM overhead by 50% vs default (sam-every=2)
t0 = time.time()
if RUN_NPO_SAM:
    !python scripts/run_npo_sam.py \
        --model {MODEL} {Q} {HF} --fast {FAST} \
        --sam-rho 0.05 --sam-every 4 \
        --run-mia \
        --output-dir {DRIVE_DIR}/npo_sam
    print(f'\n✅ NPO+SAM done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ NPO+SAM execution skipped (RUN_NPO_SAM=False). Using pre-computed results.')


[fast] Speed preset: retain=200, fp16=True, rouge_tokens=32, sam_every=4
  ARMOR — NPO + SAM (Relearning-Resistant)
  Model   : mistralai/Mistral-7B-v0.1
  Device  : cuda
  SAM ρ   : 0.05
  SAM every: 4 steps
  NPO β   : 0.1
  fp16    : True
  Retain  : 200
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:52,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:58<00:00,  4.98it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)
[model] NPO reference: using bas

In [10]:
# ── 9. RMU — Representation Misdirection  ⏱ ~4 min ──────────────────────────
t0 = time.time()
if RUN_RMU:
    !python scripts/run_rmu.py \
        --model {MODEL} {Q} {HF} {FAST} \
        --alpha 1200.0 --beta 6.5 \
        --run-mia \
        --output-dir {DRIVE_DIR}/rmu
    print(f'\n✅ RMU done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ RMU execution skipped (RUN_RMU=False). Using pre-computed results.')


  ARMOR -- RMU Unlearning
  Model : mistralai/Mistral-7B-v0.1  |  alpha=1200.0  beta=6.5
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:50,  1.00s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:59<00:00,  4.90it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)
[model] NPO reference: using base weights (LoRA-disabled mode)

[eval] Evaluating method: Pre-unlearn
[eval] Computing forget token accuracy...
[eval] Computing retain token accuracy...
               

In [11]:
# ── 10. Task Vector  ⏱ ~5 min ────────────────────────────────────────────────
t0 = time.time()
if RUN_TASK_VECTOR:
    !python scripts/run_task_vector.py \
        --model {MODEL} {Q} {HF} {FAST} \
        --lam 1.0 --run-mia \
        --output-dir {DRIVE_DIR}/task_vector
    print(f'\n✅ Task Vector done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ Task Vector execution skipped (RUN_TASK_VECTOR=False). Using pre-computed results.')

elapsed = (time.time() - NOTEBOOK_START) / 60
print(f'\n📊 Core baselines done. Elapsed: {elapsed:.0f} min')


  ARMOR -- Task Vector Unlearning
  Model : mistralai/Mistral-7B-v0.1  |  lambda=1.0
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:50,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:59<00:00,  4.88it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)

[eval] Evaluating method: Pre-unlearn
[eval] Computing forget token accuracy...
[eval] Computing retain token accuracy...
                                                            
  ARMOR Evaluation --

---
## 🔥 Section 2 — Relearning Attack (~4 min)
Tests if an adversary can fine-tune the model back to remembering forgotten data.
**Lower recovery = better unlearning.**


In [12]:
# ── 11. Relearning Attack — GA vs NPO vs NPO+SAM  ⏱ ~4 min ──────────────────
t0 = time.time()
ga_ckpt  = f"{DRIVE_DIR}/ga/ga_unlearned"
npo_ckpt = f"{DRIVE_DIR}/npo/npo_unlearned"
sam_ckpt = f"{DRIVE_DIR}/npo_sam/npo_sam_unlearned"

if os.path.exists(ga_ckpt) and os.path.exists(npo_ckpt) and os.path.exists(sam_ckpt):
    !python scripts/run_relearning_attack.py \
        --model {MODEL} {Q} {HF} \
        --compare \
        --ga-checkpoint  {ga_ckpt} \
        --npo-checkpoint {npo_ckpt} \
        --sam-checkpoint {sam_ckpt} \
        --original-acc 0.85 \
        --n-samples 30 --epochs 2 --no-save
    print(f'\n✅ Relearning Attack done in {(time.time()-t0)/60:.1f} min')
else:
    print('ℹ️ Skipping Relearning Attack: One or more baseline checkpoints are missing.')


[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)

  ARMOR — Relearning Attack Comparison: GA vs NPO vs NPO+SAM
[model] Loading checkpoint from /content/drive/MyDrive/ARMOR_outputs/ga/ga_unlearned (is_trainable=True)
Loading weights:   1% 2/291 [00:02<04:57,  1.03s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:59<00:00,  4.86it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/pyth

---
## 🔬 Section 3 — ARMOR Modules (9 methods, ~12 min)
All use `--no-save` and `ARMOR_FAST=1` for speed.


In [13]:
# ── 12. Modules Batch A: Continual · MoE · RLACE-RMU  ⏱ ~4 min ──────────────
t0 = time.time()
print('\n' + '='*60)
print('  [1/3] Continual / Lifelong Unlearning')  # FIXED & RUNS
print('='*60)
!python scripts/run_continual_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save \
    --num-cohorts 2 \
    --output-dir {DRIVE_DIR}/continual

print('\n' + '='*60)
print('  [2/3] MoE Targeted GA')
print('='*60)
!python scripts/run_moe_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save \
    --output-dir {DRIVE_DIR}/moe

print('\n' + '='*60)
print('  [3/3] RLACE-RMU Concept Erasure')
print('='*60)
!python scripts/run_rlace_rmu.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save \
    --rlace-iters 10 \
    --output-dir {DRIVE_DIR}/rlace_rmu

print(f'\n✅ Batch A done in {(time.time()-t0)/60:.1f} min')



  [1/3] Continual / Lifelong Unlearning
  ARMOR — Continual/Lifelong Machine Unlearning
  Model       : mistralai/Mistral-7B-v0.1
  FIM Masking : False (topk=30%)
  Cohorts     : 2
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:51,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:59<00:00,  4.85it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)
[main] Loaded 40 forget samples, split into 2 cohorts.
  Cohort #1: 20 samples
  Cohort #2: 20 samples
[Cont

In [14]:
# ── 13. Modules Batch B: ZK · MIA · LoRA  ⏱ ~4 min ──────────────────────────
t0 = time.time()

print('\n' + '='*60)
print('  [1/3] ZK Verification (commit-reveal)')
print('='*60)
!python scripts/run_zk_verify.py \
    --model {MODEL} {Q} {HF} --no-save \
    --probe-samples 5

print('\n' + '='*60)
print('  [2/3] Multimodal MIA Audit')
print('='*60)
!python scripts/run_multimodal_mia.py \
    --model {MODEL} {Q} {HF} --no-save

print('\n' + '='*60)
print('  [3/3] Modular LoRA Unlearning')  # FIXED & RUNS
print('='*60)
!python scripts/run_lora_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save

print(f'\n✅ Batch B done in {(time.time()-t0)/60:.1f} min')


[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:53,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [01:00<00:00,  4.82it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)

[MM-MIA] === Running MIA Audit Pre-unlearning ===

[MIA] Auditing 'Pre-unlearning' with Min-K% (k=20%)
                                                        
[MIA] Method: Pre-unlearning
[MIA] Min-K% k=20% | AUROC = 0.4338
[MIA] Verdict: ✓ VERIFIED — forget set indistinguishable from non-members
[MIA] Forget  avg score : -7.3742
[MIA] NonMe

In [15]:
# ── 14. Modules Batch C: NASD · HDI · CAS  ⏱ ~4 min ─────────────────────────
t0 = time.time()

print('\n' + '='*60)
print('  [1/3] NASD Decay')
print('='*60)
!python scripts/run_nasd.py \
    --model {MODEL} {Q} {HF} {FAST}

print('\n' + '='*60)
print('  [2/3] HDI Zero-Shot')  # FIXED & RUNS
print('='*60)
!python scripts/run_hdi_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save

print('\n' + '='*60)
print('  [3/3] CAS Graph Blockade')
print('='*60)
!python scripts/run_cas_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} --no-save

print(f'\n✅ Batch C done in {(time.time()-t0)/60:.1f} min')
print(f'📊 All ARMOR Modules done. Elapsed: {(time.time()-NOTEBOOK_START)/60:.0f} min')



  [1/3] NASD Decay

  [1/3] NASD Decay
  ARMOR Vanguard — Neuro-Apoptotic Subnetwork Decay (NASD)
  Model       : mistralai/Mistral-7B-v0.1
  Decay Steps : 50
  Decay Rate  : 0.99
  Target Top-K: 0.10%
[data] Loading TOFU — forget=forget01, retain=retain99
  ARMOR Vanguard — Neuro-Apoptotic Subnetwork Decay (NASD)
  Model       : mistralai/Mistral-7B-v0.1
  Decay Steps : 50
  Decay Rate  : 0.99
  Target Top-K: 0.10%
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<05:19,  1.11s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious. 

---
## 🛡️ Section 4 — ARMOR Extensions *(Optional — adds ~15 min)*
MultiTask-NPO · DP-NPO+SAM · LLaVA · MUSE

> **Skip this section if time is tight.** The core results above are sufficient.


In [16]:
# ── 15. Extensions (Optional)  ⏱ ~15 min total ──────────────────────────────
# Uncomment the methods you want to run:

t0 = time.time()

# --- MultiTask-NPO ---
print('\n[ext] MultiTask-NPO...')
!python scripts/run_multitask_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} \
    --n-tasks 2 --run-mia --no-save \
    --output-dir {DRIVE_DIR}/multitask_npo

# --- DP-NPO+SAM ---
print('\n[ext] DP-NPO+SAM...')
!python scripts/run_dp_armor.py \
    --model {MODEL} {Q} {HF} {FAST} \
    --epsilon 8.0 --delta 1e-5 --noise 1.0 --clip 1.0 \
    --run-mia --no-save \
    --output-dir {DRIVE_DIR}/dp_npo_sam

# --- LLaVA Cross-Modal ---
print('\n[ext] LLaVA NPO+SAM...')
!python scripts/run_llava_unlearn.py \
    --model {MODEL} {Q} {HF} {FAST} \
    --text-only --run-mia --no-save \
    --output-dir {DRIVE_DIR}/llava_npo_sam

# --- MUSE Benchmark ---
print('\n[ext] MUSE books...')
!python scripts/run_muse_benchmark.py \
    --model {MODEL} {Q} {HF} {FAST} \
    --domain books --method npo_sam \
    --run-mia --no-save \
    --output-dir {DRIVE_DIR}/muse_books

print(f'\n✅ Extensions done in {(time.time()-t0)/60:.1f} min')



[ext] MultiTask-NPO...
  ARMOR -- Multi-Task NPO Unlearning
  Model   : mistralai/Mistral-7B-v0.1  |  N-tasks: 2
[data] Loading TOFU — forget=forget01, retain=retain99
[data] Retain set (subsampled): 200 samples (max_retain_samples=200)
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
Loading weights:   1% 2/291 [00:02<04:50,  1.01s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [01:00<00:00,  4.82it/s]
trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.0470
[model] Total params     : 3755.5M
[model] Trainable params : 3.4M (0.1%)
[model] NPO reference: using base weights (LoRA-disabled mode)
[multitask] Tasks: ['Author_A', 'Author_B']
  Author_A: 20 samples
  Author_B: 20 samples

[eval] Evaluating meth

---
## 🚀 Section 5 — Phase 1-3 Frontier Research *(Optional — adds ~20 min)*
Skip if time-constrained. These are supplementary research methods.


In [ ]:
# ── 16. Phase 1-3 Research Methods (Optional)  ⏱ ~20 min ────────────────────
# Uncomment individual methods as needed:

t0 = time.time()

# Phase 1
!python scripts/run_conformal_verify.py  --model {MODEL} {Q} {HF} --no-save --alpha 0.10
!python scripts/run_cot_hme.py           --model {MODEL} {Q} {HF} {FAST} --no-save --cot-coeff 0.2 --cot-max-tokens 24
!python scripts/run_temporal_unlearn.py  --model {MODEL} {Q} {HF} {FAST} --no-save --halflife-days 1.0

# Phase 2
!python scripts/run_lcage.py                --model {MODEL} {Q} {HF} {FAST} --no-save
!python scripts/run_reconsolidation.py      --model {MODEL} {Q} {HF} {FAST} --no-save
!python scripts/run_morphogenetic_repair.py --model {MODEL} {Q} {HF} {FAST} --no-save

# Phase 3
!python scripts/run_stackelberg_game.py  --model {MODEL} {Q} {HF} {FAST} --no-save
!python scripts/run_causal_iu.py         --model {MODEL} {Q} {HF} {FAST} --no-save
!python scripts/run_federated_robust.py  --model {MODEL} {Q} {HF} {FAST} --no-save

# Bonus
!python scripts/run_reconstruction_attack.py --model {MODEL} {Q} {HF} --no-save
!python scripts/run_audit_gen.py --model {MODEL} {Q} {HF} --probe-samples 5 --output-dir {DRIVE_DIR}/audit

print(f'\n✅ Phase 1-3 done in {(time.time()-t0)/60:.1f} min')


usage: run_conformal_verify.py [-h] [--debug] [--model MODEL] [--qlora]
                               [--alpha ALPHA]
                               [--retain-check-n RETAIN_CHECK_N] [--save-html]
                               [--no-save] [--output-dir OUTPUT_DIR]
run_conformal_verify.py: error: unrecognized arguments: --hf-token hf_REDACTED_TOKEN
usage: run_conformal_verify.py [-h] [--debug] [--model MODEL] [--qlora]
                               [--alpha ALPHA]
                               [--retain-check-n RETAIN_CHECK_N] [--save-html]
                               [--no-save] [--output-dir OUTPUT_DIR]
run_conformal_verify.py: error: unrecognized arguments: --hf-token hf_REDACTED_TOKEN
usage: run_cot_hme.py [-h] [--debug] [--model MODEL] [--qlora]
                      [--cot-coeff COT_COEFF] [--cot-threshold COT_THRESHOLD]
                      [--cot-max-tokens COT_MAX_TOKENS] [--no-rouge]
                      [--no-save] [--output-dir OUTPUT_DIR]
run_cot_hme.py: error: unr

---
## 📊 Section 6 — Results & Visualisations


In [ ]:
# ── 17. Collect Results  ⏱ ~10s ──────────────────────────────────────────────
import json, os, glob
import pandas as pd

results = []

# 1. Load pre-computed baseline results from the repository if available
baseline_file = os.path.join('/content/ARMOR', 'armor/baseline_results.json')
baseline_data = {}
if os.path.exists(baseline_file):
    try:
        with open(baseline_file) as f:
            baseline_data = json.load(f)
    except Exception as e:
        print(f"Warning loading baseline cache: {e}")

# Pre-populate results with baseline_data
for method, metrics in baseline_data.items():
    results.append({
        'Method'           : method,
        'Forget Quality ↑' : round(metrics.get('forget_quality', 0), 4),
        'Forget Acc ↓'     : round(metrics.get('forget_accuracy', 0), 4),
        'Retain Acc ↑'     : round(metrics.get('retain_accuracy', 0), 4),
        'MIA AUROC →0.5'   : round(metrics.get('mia_auroc', -1), 4),
    })

# 2. Collect actually run results from DRIVE_DIR, overriding cached ones if present
for method_dir in sorted(glob.glob(f'{DRIVE_DIR}/*')):
    method_name = os.path.basename(method_dir)
    for jf in glob.glob(f'{method_dir}/**/*.json', recursive=True):
        try:
            with open(jf) as f:
                data = json.load(f)
            if 'forget_quality' in data:
                new_entry = {
                    'Method'           : method_name,
                    'Forget Quality ↑' : round(data.get('forget_quality',   0), 4),
                    'Forget Acc ↓'     : round(data.get('forget_accuracy',  0), 4),
                    'Retain Acc ↑'     : round(data.get('retain_accuracy',  0), 4),
                    'MIA AUROC →0.5'   : round(data.get('mia_auroc',       -1), 4),
                }
                # Replace if already exists in results
                existing_idx = next((i for i, r in enumerate(results) if r['Method'] == method_name), None)
                if existing_idx is not None:
                    results[existing_idx] = new_entry
                else:
                    results.append(new_entry)
                break
        except Exception:
            pass

if results:
    df = pd.DataFrame(results).sort_values('Forget Quality ↑', ascending=False)
    print('\n' + '='*72)
    print('  ARMOR — Experiment Results')
    print('='*72)
    print(df.to_string(index=False))
    df.to_csv(f'{DRIVE_DIR}/summary_results.csv', index=False)
    print(f'\n✅ Results saved to {DRIVE_DIR}/summary_results.csv')
else:
    print('No results found yet. Run experiment cells first.')


In [ ]:
# ── 18. Plot Results  ⏱ ~10s ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if results:
    df_plot = df.copy()
    methods = df_plot['Method'].tolist()
    fq      = df_plot['Forget Quality ↑'].tolist()
    ra      = df_plot['Retain Acc ↑'].tolist()
    mia     = [v if v >= 0 else 0.5 for v in df_plot['MIA AUROC →0.5'].tolist()]

    x, width = np.arange(len(methods)), 0.25
    fig, ax = plt.subplots(figsize=(max(12, len(methods)*0.8), 6))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')

    ax.bar(x - width, fq,  width, label='Forget Quality ↑',  color='#58a6ff', alpha=0.9)
    ax.bar(x,         ra,  width, label='Retain Acc ↑',       color='#3fb950', alpha=0.9)
    ax.bar(x + width, mia, width, label='MIA AUROC (→0.5)',   color='#f78166', alpha=0.9)
    ax.axhline(y=0.5, color='#f0e68c', linestyle='--', alpha=0.6, linewidth=1.2,
               label='Ideal MIA = 0.5')

    for spine in ax.spines.values():
        spine.set_color('#30363d')
    ax.tick_params(colors='#e6edf3')
    ax.set_xlabel('Unlearning Method', color='#e6edf3', fontsize=11)
    ax.set_ylabel('Score', color='#e6edf3', fontsize=11)
    ax.set_title('ARMOR — Unlearning Methods Comparison', color='#e6edf3',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=35, ha='right', color='#e6edf3', fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.legend(facecolor='#21262d', labelcolor='#e6edf3', edgecolor='#30363d')
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/results_comparison.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'✅ Plot saved to {DRIVE_DIR}/results_comparison.png')
else:
    print('No results to plot.')


In [ ]:
# ── 19. Privacy Audit Summary  ⏱ ~5s ────────────────────────────────────────
print('\n' + '='*65)
print('  ARMOR — Privacy Audit (Min-K% Prob MIA)')
print('='*65)
print('  Ideal AUROC = 0.500  →  model treats forget-set as non-members')
print('  AUROC > 0.70         →  unlearning FAILED')
print()

if results:
    for r in df.to_dict('records'):
        v = r['MIA AUROC →0.5']
        if v < 0:   status = '—  (not measured)'
        elif v <= 0.55: status = '✅ VERIFIED   (near-random)'
        elif v <= 0.65: status = '⚠️  MARGINAL   (residual memory)'
        else:           status = '❌ FAILED     (model still remembers)'
        print(f"  {r['Method']:<28} AUROC={v:.4f}  {status}")

total_time = (time.time() - NOTEBOOK_START) / 60
print(f'\n⏱  Total runtime: {total_time:.1f} min')
print('='*65)


In [ ]:
# ── 20. Saved Files  ⏱ ~5s ───────────────────────────────────────────────────
print('\n📁 Files in output directory:')
for root, dirs, files in os.walk(DRIVE_DIR):
    depth = root.replace(DRIVE_DIR, '').count(os.sep)
    if depth > 2:
        continue
    total_size = sum(os.path.getsize(os.path.join(root, f))
                     for f in files if os.path.isfile(os.path.join(root, f)))
    indent = '  ' * depth
    bname  = os.path.basename(root) or 'ARMOR_outputs'
    sz     = f'({total_size/1e6:.1f} MB)' if total_size > 0 else ''
    print(f'{indent}📂 {bname}  {sz}')

total_time = (time.time() - NOTEBOOK_START) / 60
print(f'\n⏱  Total notebook runtime: {total_time:.1f} minutes')
print('✅ Done! All outputs saved.')
